# Clasificación de Sentimiento v2 — Train / Validation / Test

Pipeline completo con **tres conjuntos** para una evaluación profesional:

| Conjunto | Tamaño | Para qué sirve |
|----------|--------|----------------|
| **Train** | 64% | Entrenar el modelo |
| **Validation** | 16% | Ajustar hiperparámetros |
| **Test** | 20% | Evaluación final (se mira **una sola vez**) |

> El Test Set es sagrado: si lo usas para elegir el modelo, estás haciendo trampa sin querer.

## 0 · Instalación de dependencias

In [ ]:
!pip install datasets nltk scikit-learn -q

## Corpus

Sube el archivo `corpus_sentimiento_reviews.json` desde tu computadora.

In [ ]:
# TODO: descomenta el código de abajo y ejecútalo para subir el corpus

# → importar la utilidad de Colab para subir archivos
# from google.colab import files

# → abrir el diálogo de subida de archivos
# uploaded = files.upload()  # selecciona corpus_sentimiento_reviews.json

# → nombre del archivo que acabas de subir
# CORPUS_FILE = 'corpus_sentimiento_reviews.json'

## 1 · Imports y configuración

In [ ]:
# TODO: descomenta el código de abajo

# → librerías necesarias para el pipeline
# import json
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
# from sklearn.metrics import classification_report
# from sklearn.model_selection import train_test_split

# → constantes del experimento (puedes cambiarlas y ver qué pasa)
# TEST_SIZE    = 0.20  # 20% del total → Test
# VAL_SIZE     = 0.20  # 20% de TrainVal → Validation  (~16% del total)
# RANDOM_STATE = 42

## 2 · Métricas manuales

Implementadas desde cero para ver qué hay detrás de `classification_report`.

In [ ]:
# TODO: descomenta el código de abajo

# → función base: cuenta TP, TN, FP, FN comparando predicciones con la realidad
# def confusion_matrix_manual(y_true, y_pred, pos_label='positivo'):
#     TP = sum(t == pos_label and p == pos_label for t, p in zip(y_true, y_pred))
#     TN = sum(t != pos_label and p != pos_label for t, p in zip(y_true, y_pred))
#     FP = sum(t != pos_label and p == pos_label for t, p in zip(y_true, y_pred))
#     FN = sum(t == pos_label and p != pos_label for t, p in zip(y_true, y_pred))
#     return TP, TN, FP, FN

# → porcentaje de aciertos sobre el total
# def accuracy(y_true, y_pred):
#     TP, TN, FP, FN = confusion_matrix_manual(y_true, y_pred)
#     return (TP + TN) / (TP + TN + FP + FN)

# → de lo que el modelo dice "positivo", cuánto es realmente positivo
# def precision(y_true, y_pred, pos_label='positivo'):
#     TP, _, FP, _ = confusion_matrix_manual(y_true, y_pred, pos_label)
#     return TP / (TP + FP) if (TP + FP) > 0 else 0.0

# → de todos los positivos reales, cuántos detectó el modelo
# def recall(y_true, y_pred, pos_label='positivo'):
#     TP, _, _, FN = confusion_matrix_manual(y_true, y_pred, pos_label)
#     return TP / (TP + FN) if (TP + FN) > 0 else 0.0

# → balance entre precision y recall en un solo número
# def f1(y_true, y_pred, pos_label='positivo'):
#     p = precision(y_true, y_pred, pos_label)
#     r = recall(y_true, y_pred, pos_label)
#     return 2 * p * r / (p + r) if (p + r) > 0 else 0.0

## FASE 1 · Carga del corpus

In [ ]:
# TODO: descomenta el código de abajo

# → abrir el JSON y extraer textos y etiquetas
# with open(CORPUS_FILE, encoding='utf-8') as f:
#     data = json.load(f)

# textos_todos = [item['texto']       for item in data['reviews']]
# y_todos      = [item['sentimiento'] for item in data['reviews']]

# → verificar que el corpus se cargó correctamente
# print(f'Total de ejemplos : {len(textos_todos):,}')
# print(f'Positivos         : {y_todos.count("positivo"):,}')
# print(f'Negativos         : {y_todos.count("negativo"):,}')
# print(f'Ejemplo           : "{textos_todos[0][:80]}..."')

## FASE 2 · Split Train / Validation / Test

Se hacen dos cortes sobre el dataset completo para obtener los tres conjuntos:

```
corpus_sentimiento_reviews.json  (100%)
    ↓ primer corte: reservar 20% como Test
    ↓ segundo corte: del 80% restante, 20% → Validation
    ↓
    Train (~64%)  +  Validation (~16%)  +  Test (20%)
```

`stratify` mantiene la proporción de clases en los tres conjuntos.

In [ ]:
# TODO: descomenta el código de abajo

# → Paso 1: reservar el conjunto de Test (no se vuelve a tocar hasta la Fase 4)
# textos_trainval, textos_test, y_trainval, y_test = train_test_split(
#     textos_todos, y_todos,
#     test_size=TEST_SIZE,
#     random_state=RANDOM_STATE,
#     stratify=y_todos
# )

# → Paso 2: dividir el resto en Train + Validation
# textos_train, textos_val, y_train, y_val = train_test_split(
#     textos_trainval, y_trainval,
#     test_size=VAL_SIZE,
#     random_state=RANDOM_STATE,
#     stratify=y_trainval
# )

# → imprimir tamaños para confirmar la proporción
# total = len(textos_todos)
# print(f'Total      : {total:,} ejemplos')
# print(f'Train      : {len(textos_train):,} ejemplos  (~{len(textos_train)/total:.0%})')
# print(f'Validation : {len(textos_val):,} ejemplos  (~{len(textos_val)/total:.0%})')
# print(f'Test       : {len(textos_test):,} ejemplos  (~{len(textos_test)/total:.0%})  ← no se toca hasta Fase 4')

## FASE 3 · Búsqueda de hiperparámetros en Validation

Probamos 6 combinaciones de `ngram_range` y `max_features`.  
El criterio de selección es el **F1 en Validation** — el Test nunca interviene aquí.

In [ ]:
# TODO: descomenta el código de abajo

# → lista de combinaciones de hiperparámetros a probar
# REJILLA = [
#     {'ngram_range': (1, 1), 'max_features':  500},
#     {'ngram_range': (1, 1), 'max_features': 1000},
#     {'ngram_range': (1, 1), 'max_features': 5000},
#     {'ngram_range': (1, 2), 'max_features':  500},
#     {'ngram_range': (1, 2), 'max_features': 1000},
#     {'ngram_range': (1, 2), 'max_features': 5000},
# ]

# print(f"{'ngram_range':<15} {'max_features':<15} {'F1 (val)':>10}")
# print('-' * 42)

# mejor_f1, mejor_config = -1.0, None

# → iterar cada combinación: vectorizar, entrenar y evaluar en Validation
# for config in REJILLA:
#     vec = TfidfVectorizer(ngram_range=config['ngram_range'],
#                           max_features=config['max_features'], min_df=2)
#     X_tr  = vec.fit_transform(textos_train)
#     X_val = vec.transform(textos_val)

#     clf = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
#     clf.fit(X_tr, y_train)

#     f1_val = f1(y_val, clf.predict(X_val))
#     print(f"{str(config['ngram_range']):<15} {config['max_features']:<15} {f1_val:>10.4f}")

#     if f1_val > mejor_f1:
#         mejor_f1, mejor_config = f1_val, config

# → mostrar la mejor combinación encontrada
# print(f"\nMejor: ngram_range={mejor_config['ngram_range']}  "
#       f"max_features={mejor_config['max_features']}  F1={mejor_f1:.4f}")

## FASE 4 · Entrenamiento final y evaluación en Test

1. Reentrenamos con **Train + Validation combinados** (más datos = mejor modelo)
2. Evaluamos en **Test** por primera y única vez

In [ ]:
# TODO: descomenta el código de abajo

# → combinar Train + Validation para el entrenamiento final
# textos_train_val = textos_train + textos_val
# y_train_val      = y_train + y_val

# → vectorizar con la mejor configuración encontrada en Fase 3
# vec_final = TfidfVectorizer(ngram_range=mejor_config['ngram_range'],
#                              max_features=mejor_config['max_features'], min_df=2)
# X_train_val = vec_final.fit_transform(textos_train_val)
# X_test      = vec_final.transform(textos_test)

# → entrenar el modelo final
# clf_final = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
# clf_final.fit(X_train_val, y_train_val)

# → evaluar en Test (primera y única vez)
# y_pred = clf_final.predict(X_test)

# print('Classification report (Test Set):')
# print(classification_report(y_test, y_pred))

In [ ]:
# TODO: descomenta el código de abajo

# → calcular métricas manuales sobre el Test Set
# TP, TN, FP, FN = confusion_matrix_manual(y_test, y_pred)
# print('Metricas manuales (Test Set):')
# print(f'  Matriz de confusion: TP={TP}  TN={TN}  FP={FP}  FN={FN}')
# print(f'  Accuracy  : {accuracy(y_test, y_pred):.4f}')
# print(f'  Precision : {precision(y_test, y_pred):.4f}')
# print(f'  Recall    : {recall(y_test, y_pred):.4f}')
# f1_test = f1(y_test, y_pred)
# print(f'  F1-score  : {f1_test:.4f}')

## FASE 5 · Predicción sobre ejemplos nuevos

In [ ]:
# TODO: descomenta el código de abajo

# → ejemplos nuevos que el modelo nunca vio (puedes cambiarlos)
# EJEMPLOS = [
#     'Me encanto, muy buen producto, lo recomiendo',
#     'Pesimo, llego roto y el servicio fue horrible',
#     'Calidad aceptable, funciona bien por el precio',
#     'No lo compren, una completa decepcion',
# ]

# → vectorizar y predecir con el modelo final
# preds = clf_final.predict(vec_final.transform(EJEMPLOS))

# → imprimir cada predicción junto al texto
# for texto, pred in zip(EJEMPLOS, preds):
#     print(f'  [{pred.upper():<9}]  {texto}')

## Resumen del pipeline

In [ ]:
# TODO: descomenta el código de abajo

# → imprimir resumen con la configuración ganadora y resultados finales
# print('=' * 55)
# print('RESUMEN')
# print('=' * 55)
# print(f"  Mejor config  : ngram_range={mejor_config['ngram_range']}  "
#       f"max_features={mejor_config['max_features']}")
# print(f"  F1 Validation : {mejor_f1:.4f}")
# print(f"  F1 Test       : {f1_test:.4f}")
# print()
# print('  corpus_sentimiento_reviews.json  (100%)')
# print('       | 20% → Test  (se mira una sola vez)')
# print('       | 80% → 80% Train + 20% Validation')
# print('       | rejilla hiperparametros (F1 en Validation)')
# print('       | mejor config → Train + Validation → fit final')